# Item similarity candidate search


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from itertools import product
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

CWD = Path.cwd()
ROOT_DIR = CWD if (CWD / "src").exists() else CWD.parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit
from src.models.recommender import Recommender

DATA_DIR = ROOT_DIR / "notebook" / "data" / "11092026"
if not DATA_DIR.exists():
    DATA_DIR = CWD / "data" / "11092026"


## Data


In [ ]:
df_interactions = pd.read_parquet(DATA_DIR / "interactions.parquet")
encoders = joblib.load(DATA_DIR / "encoders.pkl")
metadata = joblib.load(DATA_DIR / "metadata.pkl")

user_encoder = encoders["user_encoder"]
movie_encoder = encoders["movie_encoder"]
n_users = metadata["n_users"]

df_interactions = df_interactions.copy()
df_interactions["user_id"] = user_encoder.inverse_transform(df_interactions["user_idx"])
df_interactions["movie_id"] = movie_encoder.inverse_transform(df_interactions["movie_idx"])

splitter = TemporalSplit()
train, val, test = splitter.split_to_three(df_interactions)
rank_history, rank_target = splitter.split_to_two(train)


## Item similarity


In [ ]:
def fit_item_similarity(
    history,
    n_users,
    k_neighbors=100,
):
    movie_ids = history["movie_id"].unique()

    movie_to_sim = {
        movie_id: idx
        for idx, movie_id in enumerate(movie_ids)
    }

    sim_to_movie = {
        idx: movie_id
        for movie_id, idx in movie_to_sim.items()
    }

    positive = history.loc[
        history["rating"] >= 4,
        ["user_idx", "movie_id", "rating"]
    ].copy()

    positive["sim_movie_idx"] = positive["movie_id"].map(movie_to_sim)
    values = (positive["rating"] - 3).values.astype(np.float32)

    user_movie = csr_matrix(
        (
            values,
            (positive["user_idx"], positive["sim_movie_idx"])
        ),
        shape=(n_users, len(movie_to_sim)),
        dtype=np.float32
    )

    movie_user = normalize(user_movie.T.tocsr(), axis=1)

    model = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )

    model.fit(movie_user)

    n_neighbors = min(k_neighbors + 1, movie_user.shape[0])

    distances, indices = model.kneighbors(
        movie_user,
        n_neighbors=n_neighbors,
    )

    neighbors = {
        movie_idx: (
            indices[movie_idx][1:],
            1.0 - distances[movie_idx][1:]
        )
        for movie_idx in range(movie_user.shape[0])
    }

    return {
        "neighbors": neighbors,
        "movie_to_sim": movie_to_sim,
        "sim_to_movie": sim_to_movie,
    }


In [ ]:
def build_candidates(users, history, sim_artifacts, k=500):
    seen = history.groupby("user_idx")["movie_id"].apply(set).to_dict()

    neighbors = sim_artifacts["neighbors"]
    movie_to_sim = sim_artifacts["movie_to_sim"]
    sim_to_movie = sim_artifacts["sim_to_movie"]

    rows = []

    liked_recent = (
            history.loc[history["rating"] >= 4]
            .sort_values(["user_idx", "timestamp"])
            .groupby("user_idx")
            .tail(50)
            .groupby("user_idx")["movie_id"]
            .apply(list)
            .to_dict()
        )

    for user_idx in users:

        watched = seen.get(user_idx, set())

        sim_scores = {}
        
        for seed_movie_id in liked_recent.get(user_idx, []):
            seed_idx = movie_to_sim.get(seed_movie_id)
            if seed_idx is None:
                continue

            neighbor_ids, neighbor_scores = neighbors[seed_idx]

            for neighbor_idx, score in zip(neighbor_ids, neighbor_scores):
                movie_id = sim_to_movie[neighbor_idx]

                if movie_id in watched:
                    continue

                sim_scores[movie_id] = max(sim_scores.get(movie_id, 0), score)

        sim_top = sorted(sim_scores.items(), key=lambda x: x[1], reverse=True)[:k]

        for movie_id, score in sim_top:
            rows.append(
                {
                    "user_idx": user_idx,
                    "movie_id": movie_id,
                    "score": score
                }
            )

    result = pd.DataFrame(rows)
    result["user_id"] = user_encoder.inverse_transform(result["user_idx"])

    return result


In [ ]:
def recall(candidates, target, min_rating=4):

    relevant = (
        target.loc[target["rating"] >= 4]
        .groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    predicted = candidates.groupby("user_id")["movie_id"].apply(set).to_dict()

    recalls = []

    for user_id, y_true in relevant.items():
        if not y_true:
            continue

        y_pred = predicted.get(user_id, set())
        recalls.append(len(y_true & y_pred) / len(y_true))

    return np.mean(recalls) if recalls else 0.0


## Similarity baseline


In [ ]:
sim_artifacts = fit_item_similarity(rank_history, n_users, k_neighbors=600)
sim_candidates = build_candidates(
    rank_target["user_idx"].unique(),
    rank_history,
    sim_artifacts,
    k=500
)
print("Similarity Recall@500:", recall(sim_candidates, rank_target))


## Retrieval source search


In [ ]:
als_candidates = joblib.load(DATA_DIR / "als_candidates.pkl")

sim_ks = [800, 900, 1000, 1100]
pop_ks = [0, 100, 300, 600]
MAX_SIM_K = max(sim_ks)
MAX_POP_K = max(pop_ks)

sim_artifacts = fit_item_similarity(rank_history, n_users, k_neighbors=600)
all_sim_candidates = (
    build_candidates(
        rank_target["user_idx"].unique(),
        rank_history,
        sim_artifacts,
        k=MAX_SIM_K
    )
    .sort_values(["user_id", "score"], ascending=[True, False])
)

pop_model = Recommender().fit(rank_history)
user_mapping = rank_target[["user_idx", "user_id"]].drop_duplicates()

pop_rows = []
for user_idx, user_id in user_mapping.itertuples(index=False):
    for movie_id, score in pop_model.recommend(user_id=user_idx, k=MAX_POP_K):
        pop_rows.append((user_id, movie_id, score))

all_pop_candidates = (
    pd.DataFrame(pop_rows, columns=["user_id", "movie_id", "score"])
    .sort_values(["user_id", "score"], ascending=[True, False])
)


In [ ]:
results = []

for sim_k, pop_k in product(sim_ks, pop_ks):
    sim_part = all_sim_candidates.groupby("user_id", sort=False).head(sim_k)
    pop_part = (
        all_pop_candidates.iloc[0:0]
        if pop_k == 0
        else all_pop_candidates.groupby("user_id", sort=False).head(pop_k)
    )

    full_candidates = (
        pd.concat([
            als_candidates[["user_id", "movie_id"]],
            sim_part[["user_id", "movie_id"]],
            pop_part[["user_id", "movie_id"]]
        ])
        .drop_duplicates(["user_id", "movie_id"])
    )

    results.append({
        "als_k": 500,
        "sim_k": sim_k,
        "pop_k": pop_k,
        "mean_candidates": full_candidates.groupby("user_id").size().mean(),
        "recall": recall(full_candidates, rank_target)
    })

results_df = pd.DataFrame(results).sort_values("recall", ascending=False).reset_index(drop=True)
results_df


## Selected retrieval


In [ ]:
K_ALS = 500
K_SIM = 1100
K_POP = 300

print(K_ALS, K_SIM, K_POP)
